# flash-attention-cuda — live demo

Runs the whole verification story on a free Colab GPU (Runtime → Change runtime type → **T4 GPU**):
the CPU algorithm tests, the hand-written CUDA kernel against the float64 reference,
the Triton rendering of the same algorithm, and a timing table of all of them vs PyTorch SDPA.

Repo: https://github.com/agupt0318/flash-attention-cuda

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime → Change runtime type → T4 GPU, then rerun'
!pip install -q ninja        # torch's C++ extension JIT requires it; Colab doesn't ship it
![ -d flash-attention-cuda ] || git clone -q https://github.com/agupt0318/flash-attention-cuda.git
%cd flash-attention-cuda
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. The algorithm, validated on CPU (no GPU involved)

In [ ]:
!make test

## 2. The CUDA kernel vs the reference
Compiles for this machine's GPU, then checks the usual hostile shapes at 5e-5.

In [ ]:
import torch
arch = 'sm_%d%d' % torch.cuda.get_device_capability()
!make test-gpu ARCH={arch}

## 3. PyTorch wrapper — parity vs SDPA, float64 as judge

In [ ]:
!python3 pytorch/test_parity.py bench

## 4. The Triton rendering of the same algorithm

In [ ]:
!python3 triton/test_parity.py

## 5. Everyone on one axis
CUDA kernel vs Triton vs SDPA vs **naive attention that materializes the N×N matrix in HBM** — the last one is the baseline the paper's claims are about. At N=4096 this shape's naive intermediates are ~2 GB per matmul; watch its curve (and its memory) diverge.

In [ ]:
import os, sys
# self-locate: survives runtime restarts and out-of-order cell runs
for d in ('flash-attention-cuda', '.', '..'):
    if os.path.isdir(os.path.join(d, 'pytorch')) and \
       os.path.isfile(os.path.join(d, 'Makefile')):
        os.chdir(d)
        break
else:
    raise RuntimeError('repo not found — run the setup cell at the top first')
sys.path[:0] = [os.path.abspath('pytorch'), os.path.abspath('triton')]

import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from flash_attn import flash_attention
from flash_attn_triton import flash_attention_triton

def naive(q, k, v):                     # the paper's baseline: N^2 in HBM
    s = q @ k.transpose(-1, -2) / (q.size(-1) ** 0.5)
    return s.softmax(-1) @ v

def time_ms(fn, iters=30):
    fn(); torch.cuda.synchronize()
    t0, t1 = torch.cuda.Event(True), torch.cuda.Event(True)
    t0.record()
    for _ in range(iters): fn()
    t1.record(); torch.cuda.synchronize()
    return t0.elapsed_time(t1) / iters

impls = {'cuda (ours)': flash_attention,
         'triton (ours)': flash_attention_triton,
         'torch SDPA': lambda q, k, v: F.scaled_dot_product_attention(q, k, v),
         'naive (N² in HBM)': naive}
seqs, results = [512, 1024, 2048, 4096], {n: [] for n in impls}
for N in seqs:
    q, k, v = (torch.randn(4, 8, N, 64, device='cuda') for _ in range(3))
    for name, fn in impls.items():
        torch.cuda.reset_peak_memory_stats()
        ms = time_ms(lambda: fn(q, k, v),
                     iters=10 if 'naive' in name else 30)
        results[name].append(ms)
        peak = torch.cuda.max_memory_allocated() / 2**30
        print(f'N={N:<5} {name:18} {ms:8.3f} ms   peak mem {peak:5.2f} GiB')

for name, ys in results.items(): plt.plot(seqs, ys, marker='o', label=name)
plt.xscale('log', base=2); plt.yscale('log')
plt.xlabel('sequence length'); plt.ylabel('ms (fwd)'); plt.legend()
plt.title('B=4 H=8 d=64, fp32'); plt.show()

## 6. Profile the kernel (optional but decisive)
Nsight Compute's speed-of-light section on one N=4096 launch: SM vs memory utilization and the top stall reasons — the difference between guessing at bottlenecks and reading them. Works on Colab because the VM runs as root; if you see `ERR_NVGPUCTRPERM`, the platform blocks counters and this cell can be skipped.

In [ ]:
import os, shutil, subprocess, torch
for d in ('flash-attention-cuda', '.', '..'):
    if os.path.isfile(os.path.join(d, 'Makefile')):
        os.chdir(d)
        break
arch = 'sm_%d%d' % torch.cuda.get_device_capability()
ncu = shutil.which('ncu') or '/usr/local/cuda/bin/ncu'
subprocess.run(['make', f'ARCH={arch}', 'build/bench'],
               stdout=subprocess.DEVNULL, check=True)
# bench launches 62 kernels per seq length (2 masks x (1 warmup + 30));
# skip 190 lands inside the N=4096 non-causal timed region
out = subprocess.run(
    [ncu, '--kernel-name', 'regex:flash_fwd', '--launch-skip', '190',
     '--launch-count', '1', '--section', 'SpeedOfLight',
     '--section', 'SchedulerStats', './build/bench'],
    capture_output=True, text=True).stdout
tail = out[out.find('flash_fwd'):] if 'flash_fwd' in out else out
print('\n'.join(tail.splitlines()[:50]) or 'ncu produced no output — counters may be blocked')